# Table 3 Scalability Analysis (160x160 World)

This notebook parses scalability experiment logs for **Hybrid Standard** and **ODRM\*** planners.

For each agent size (250, 500, 750, 1000), we compute:
- **Success Rate (%)**
- **Average Agent Completion Rate (%)**

The results are summarized per model and compared in a final table.

In [1]:
import re
import pandas as pd
from collections import defaultdict

In [2]:
files = {
    "Hybrid Standard": "Results_table_3/Hybrid_standard_scalability_test_size160x160.txt",
    "ODRM*": "Results_table_3/Odrmstar_scalability_test_size160x160.txt"
}

In [3]:
def parse_scalability(file_path):

    data = defaultdict(lambda: {"success":0, "total":0, "completion_rates":[]})

    with open(file_path, "r") as f:
        current_agents = None

        for line in f:

            if line.startswith("Started Experiment"):
                match = re.search(r"Number of agents:\s*(\d+)", line)
                if match:
                    current_agents = int(match.group(1))

            if line.startswith("[EPISODE"):

                data[current_agents]["total"] += 1

                if "Success: True" in line:
                    data[current_agents]["success"] += 1

                comp_match = re.search(r"Agents Completed:\s*(\d+)/(\d+)", line)

                if comp_match:
                    completed = int(comp_match.group(1))
                    total_agents = int(comp_match.group(2))

                    completion_rate = (completed / total_agents) * 100
                    data[current_agents]["completion_rates"].append(completion_rate)

    rows = []

    for agents in sorted(data.keys()):

        success_rate = (data[agents]["success"] / data[agents]["total"]) * 100 if data[agents]["total"] else 0

        avg_completion = sum(data[agents]["completion_rates"]) / len(data[agents]["completion_rates"])

        rows.append({
            "Agents": agents,
            "Success Rate (%)": success_rate,
            "Avg Completion (%)": avg_completion
        })

    return pd.DataFrame(rows)

In [4]:
import re
import pandas as pd
from collections import defaultdict

def parse_odrm(file_path):

    data = defaultdict(lambda: {"success":0, "total":0, "oom":False})
    current_agents = None

    with open(file_path, "r") as f:
        for line in f:

            if "Agents:" in line:
                match = re.search(r"Agents:\s*(\d+)", line)
                if match:
                    current_agents = int(match.group(1))

            if line.startswith("[EPISODE"):

                data[current_agents]["total"] += 1

                if "Success: True" in line:
                    data[current_agents]["success"] += 1

                if "std::bad_alloc" in line:
                    data[current_agents]["oom"] = True

    rows = []

    for agents in sorted(data.keys()):

        if data[agents]["oom"]:
            success_rate = "OOM"
            completion = "OOM"
        else:
            success_rate = (data[agents]["success"] / data[agents]["total"]) * 100

            if success_rate == 100:
                completion = 100.0
            else:
                completion = "NA"

        rows.append({
            "Agents": agents,
            "Success Rate (%)": success_rate,
            "Avg Completion (%)": completion
        })

    return pd.DataFrame(rows)

In [5]:
hybrid_df = parse_scalability(files["Hybrid Standard"])
odrm_df = parse_odrm(files["ODRM*"])

In [6]:
comparison = hybrid_df.copy()

comparison["ODRM* Success (%)"] = odrm_df["Success Rate (%)"]
comparison["ODRM* Completion (%)"] = odrm_df["Avg Completion (%)"]


comparison.columns = [
    "Agents",
    "Hybrid Success (%)",
    "Hybrid Completion (%)",
    "ODRM* Success (%)",
    "ODRM* Completion (%)"
]

comparison

,Agents,Hybrid Success (%),Hybrid Completion (%),ODRM* Success (%),ODRM* Completion (%)
0,250,0.0,78.266667,100.0,100.0
1,500,0.0,82.133333,OOM,OOM
2,750,0.0,83.911111,OOM,OOM
3,1000,0.0,83.200000,OOM,OOM
